# Práctico 2 - Aplicación de LLMs

### Angaut, Gonzalo

Para este práctico, se pide:

1. Cargar un SLM (Small Language Model) y hacerlo funcionar a modo de pregunta respuesta como lo haria ChatGPT -> Una opcion puede ser [Qwen de Alibaba](https://huggingface.co/Qwen/Qwen3-0.6B) u otro modelo similar con < 1 Bi de parámetros.

1. Cargar el dataset elegido en el Práctico 1 e iterar cada una de las filas para generar una predicción.

1. Cargar el modelo entrenado en el Práctico 1 y hacer una comparación ¿Es mejor el LLM para clasificar? ¿Por qué?

1. Iterar parametros y prompt para ver como mejora.

1. Finetunear el SLM (Opcional).

1. Incorporar un modelo de huggingface (ej. BERT) a la comparación (Opcional).

La notebook a presentar debe ser legible incluyendo.
1. Introducción

1. Acompañar con comentarios que aporten a la interpretación de los resultados.

1. Una conclusión (breve pero no tan breve) con un resumen de lo trabajado y los resultados más representativos de acuerdo a su interpretación


In [1]:
import json
import requests

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import re

from sklearn.model_selection import train_test_split

from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Preparamos el terreno

In [2]:
!pip install transformers

Importamos e instanciamos

## Introducción

En el presente trabajo práctico, exploraremos las capacidades de los Modelos de Lenguaje Pequeños (SLMs), una variante de menor escala de los Modelos de Lenguaje Grandes (LLMs), para la tarea de clasificación de sentimiento.

Al comienzo, cargaremos un SLM pre-entrenado, el Qwen3-0.6B, y lo configuraremos para operar como un chatbot interactivo en modo de pregunta y respuesta.

Posteriormente, utilizaremos el dataset de reseñas de películas IMDB (empleado en el Práctico 1) para realizar una clasificación de sentimiento en modo zero-shot y few-shot, utilizando técnicas de Prompt Engineering y Generación Determinística.

El objetivo central de esta práctica es comparar y analizar el rendimiento del SLM frente al modelo tradicional de TF-IDF entrenado en el Práctico 1. Además, se iterarán los parámetros de inferencia (num_beams, max_new_tokens) y el diseño del prompt para evaluar su impacto en la precisión, robustez y velocidad de la clasificación.

## Desarrollo del trabajo

Para este trabajo, vamos a utilizar el [Qwen de Alibaba](https://huggingface.co/Qwen/Qwen3-0.6B), una familia de Modelos de Lenguaje Grandes (LLMs) y Modelos de Lenguaje Pequeños (SLMs) desarrollados por Alibaba Cloud.

Cargamos este SLM:

In [3]:
model_name = "Qwen/Qwen3-0.6B"

# Cargamos el tokenizer y el modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

# Vemos el dispositivo que estamos usando
device = model.device
print(f"Modelo cargado en el dispositivo: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo cargado en el dispositivo: cuda:0


A este modelo lo vamos a hacer funcionar a modo de pregunta respuesta como lo haria ChatGPT.

In [4]:
def responder_mensajes(messages, model, tokenizer):
  # Aplicar plantilla de chat
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  # Tokenizar entrada
  inputs = tokenizer([text], return_tensors="pt").to(model.device)

  # Generar respuesta
  outputs = model.generate(
      **inputs,
      max_new_tokens=256,
      do_sample=True,# En false es deterministico
      top_p=0.9,
      temperature=0.7,
      pad_token_id=tokenizer.eos_token_id # Para evitar warnings
  )

  # Decodificar solo la parte nueva generada
  response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

  # Vamos a usar expresiones para eliminar el bloque <think>
  cleaned_response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

  # Devolvemos el texto limpio para que pueda agregarse al historial si es un chat interactivo
  return cleaned_response

Entonces, podemos hablar con el chat definiendo un mensaje:

In [5]:
# Prompt estilo chat
messages = [
    {"role": "user", "content": "Dame una breve introducción a los modelos de lenguaje grandes."}
]

respuesta = responder_mensajes(messages, model, tokenizer)
print("Assistant:", respuesta)

Assistant: Los modelos de lenguaje grandes son sistemas de inteligencia artificial diseñados para entender, generar y procesar textos de manera eficiente. Estos modelos, como GPT, BERT o T5, son capaces de procesar grandes textos, generar contenido, y realizar tareas como traducción, resumen, y análisis de datos. Su desarrollo se basa en datos de entrenamiento, lo que les da capacidades profundas en lenguaje natural y contexto. Estos modelos son esenciales para la inteligencia artificial moderna, permitiendo aplicaciones en áreas como la comunicación, la tecnología y la investigación.


Podemos hacer para que sea un chat interactivo:

In [6]:
def chat_interactivo(model, tokenizer):
    print("CHAT")
    print("Escribe 'salir' para terminar la conversación.")

    # Inicializamos una lista de mensajes para mantener el historial
    # Esto permite que el modelo "recuerde" conversaciones anteriores.
    # Además, le vamos a decir que conteste en español
    conversation_history = [
        {"role": "system", "content": "Eres un asistente de inteligencia artificial que siempre debe responder en ESPAÑOL."}
    ]

    while True:
        user_input = input("You: ")

        if user_input.lower() == 'salir':
            print("Chat terminado.")
            break

        # Añadimos el nuevo mensaje del usuario al historial
        conversation_history.append({"role": "user", "content": user_input})

        # Llamamos a la función, pasándole el historial completo
        assistant_response = responder_mensajes(conversation_history, model, tokenizer)

        # Añadimos la respuesta limpia del asistente al historial para mantener el contexto
        conversation_history.append({"role": "assistant", "content": assistant_response})

        print("Assistant:", assistant_response)

Y entonces si llamamos a la función, podemos tener una conversación:

In [7]:
chat_interactivo(model, tokenizer)

CHAT
Escribe 'salir' para terminar la conversación.
You: ¡Hola! ¿Como estas?
Assistant: Hola! ¿Cómo estás? Soy un asistente de inteligencia artificial, y estoy aquí para ayudarte. ¿Te puedo ayudar con algo?
You: Si, quería saber que opinas de la milanesa napolitana
Assistant: Hola, soy un asistente de inteligencia artificial y no tengo opiniones personales. Si quieres, puedo ayudarte a entender más sobre el Napolitana milanesa o cualquier otra información! ¿Te gustaría saber algo más?
You: Si, quiero saber más sobre la milanesa napolitana
Assistant: La milanesa napolitana es una típica comida italiana, específicamente una receta
You: salir
Chat terminado.


Ahora vamos a cargar el dataset elegido en el práctico 1, que fue el de IMDB:

In [8]:
# Remueve archivo anterior (si existe)
!rm -f imdb_dataset.csv

# Usa wget para descarga
file_id = "1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3"
!wget --no-check-certificate "https://docs.google.com/uc?export=download&id={file_id}" -O imdb_dataset.csv

# Carga el dataset
with open("imdb_dataset.csv", 'r') as file:
    df_imdb = pd.read_csv(file)
    print(f"Success! Dataset loaded with {len(df_imdb)} records.")

--2025-10-19 17:05:48--  https://docs.google.com/uc?export=download&id=1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3
Resolving docs.google.com (docs.google.com)... 172.253.118.139, 172.253.118.101, 172.253.118.102, ...
Connecting to docs.google.com (docs.google.com)|172.253.118.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3&export=download [following]
--2025-10-19 17:05:49--  https://drive.usercontent.google.com/download?id=1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 64.233.170.132, 2404:6800:4003:c1a::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|64.233.170.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66212309 (63M) [application/octet-stream]
Saving to: ‘imdb_dataset.csv’

imdb_dataset.csv    100%[===================>]  63.14M  42.3MB/

In [9]:
df = df_imdb
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Veamos, por ejemplo, una crítica:

In [10]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

Ahora debemos ver cómo generar una predicción. Para eso, debemos armar un prompt.

Vamos a hacer una función que clasifique texto usando el SLM.

In [11]:
def classify_slm(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS (FEW-SHOT) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Veamos por ejemplo como clasifica algun comentario:

In [13]:
i=10
text_to_classify = df['review'][i]
prediction = classify_slm(text_to_classify, model, tokenizer, device)

print(f"Comentario (Índice {i}): {text_to_classify[:100]}...")
print(f"Predicción estable del SLM: {prediction}")
print(f"Etiqueta real: {df['sentiment'][i]}")

Comentario (Índice 10): Phil the Alien is one of those quirky films where the humour is based around the oddness of everythi...
Predicción estable del SLM: negative
Etiqueta real: negative


Podemos notar como este comentario en particular lo clasifica correctamente!

Ahora iteremos y hagamos sobre muchas filas, en particular elegiremos 1000 filas al azar. Para esto, creemos una función:

In [15]:
def iterate_classification(df, classify_fn, model, tokenizer, device, n_samples=1000):

  # Creamos un dataset con n_samples muestras
  df_sample = df.sample(n=n_samples, random_state=42).copy()
  predictions = []

  print(f"Iniciando la predicción del SLM para {n_samples} muestras...")

  # Iteramos sobre el DataFrame de muestra con TQDM para seguimiento
  for index, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
      review_text = row['review']

      # Llamamos a la función de clasificación
      prediction = classify_fn(review_text, model, tokenizer, device)
      predictions.append(prediction)

  # Agregamos las predicciones al DataFrame de muestra
  df_sample['prediction'] = predictions

  return df_sample

Y ahora llamamos a la función:

In [16]:
df_sample = iterate_classification(df, classify_slm, model, tokenizer, device, n_samples=1000)

Iniciando la predicción del SLM para 1000 muestras...


100%|██████████| 1000/1000 [16:51<00:00,  1.01s/it]


Veamos algunas críticas junto con el sentimiento y las predicciones generadas:

In [18]:
print("\n--- Predicciones del SLM Generadas ---")
print(df_sample[['review', 'sentiment', 'prediction']].sample(10))


--- Predicciones del SLM Generadas ---
                                                  review sentiment  prediction
37915  I haven't seen this film for over 20 years, bu...  positive    positive
47062  This film is more about how children make sens...  positive    positive
31469  It is not the same as the other films about da...  positive    positive
41514  If there was some weird inversed Oscar Academy...  negative    positive
26546  National Lampoon's Christmas Vacation 2: Cousi...  negative  INDEFINIDO
4347   Some people might consider this movie a piece ...  negative    negative
22319  Lana Turner proved that she could really dance...  positive    positive
38064  by Dane Youssef<br /><br />A gang of crooks. T...  negative    negative
23921  During university, our Philosophy professor, M...  negative    positive
46981  The cast was good, and I thought it was a good...  positive    positive


Por último, calculemos las métricas. Para eso armamos la función:

In [19]:
def evaluate_slm_predictions(df, true_col='sentiment', pred_col='prediction', undefined_label='INDEFINIDO', verbose=True):
    # Filtramos los casos válidos
    df_cleaned = df[df[pred_col] != undefined_label].copy()

    # Convertimos etiquetas reales a minúsculas
    y_true = df_cleaned[true_col].apply(lambda x: x.lower())
    y_pred = df_cleaned[pred_col]

    # Métricas de conteo
    total_samples = len(df)
    valid_samples = len(df_cleaned)
    failed_samples = total_samples - valid_samples

    # Calculamos métricas
    acc = accuracy_score(y_true, y_pred)
    report_dict = classification_report(y_true, y_pred, output_dict=True)
    f1_score_slm = report_dict['weighted avg']['f1-score']

    # Mostramos resultados
    if verbose:
        print(f"Total de muestras evaluadas: {total_samples}")
        print(f"Muestras con FALLO de instrucción ('{undefined_label}'): {failed_samples} ({failed_samples / total_samples * 100:.2f}%)")
        print("-" * 35)
        print(f"Precisión (Accuracy): {acc:.4f}")
        print(f"F1-score ponderado: {f1_score_slm:.4f}")
        print("\nReporte completo:")
        print(classification_report(y_true, y_pred, digits=4))

    # Retornamos resultados como diccionario
    return {
        "accuracy": acc,
        "f1_weighted": f1_score_slm,
        "total_samples": total_samples,
        "valid_samples": valid_samples,
        "failed_samples": failed_samples,
        "report": report_dict
    }

Y ahora llamamos a la función:

In [20]:
results = evaluate_slm_predictions(df_sample)

Total de muestras evaluadas: 1000
Muestras con FALLO de instrucción ('INDEFINIDO'): 95 (9.50%)
-----------------------------------
Precisión (Accuracy): 0.7923
F1-score ponderado: 0.7895

Reporte completo:
              precision    recall  f1-score   support

    negative     0.9273    0.6618    0.7724       482
    positive     0.7094    0.9409    0.8089       423

    accuracy                         0.7923       905
   macro avg     0.8184    0.8014    0.7907       905
weighted avg     0.8255    0.7923    0.7895       905



## Comparación con resultados previos

Si recordamos del trabajo anterior, el enfoque de representación que mejor andaba en la clasificación posterior era TF-IDF con una precisión en el conjunto de testeo de $\approx 0.90$ y un F1 de $\approx 0.89$. Estos resultados son mejores que los obtenidos por el modelo base con SLM, aunque este igualmente clasifica con unas métricas bastante altas. Por lo tanto, SLM, sin entrenamiento, es ligeramente inferior a las técnicas tradicionales, pero al iterar prompts y parámetros y hacer fine-tuning, esto debería mejorar.

## Iteración de parámetros y prompts

Ahora iteremos los parámetros y prompts a ver si conseguimos mejoras.

Por ejemplo, agreguemos más ejemplos.

In [21]:
def classify_slm_2(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS EXTENDIDOS (4 Ejemplos) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"

            "Comentario: 'Una pérdida de tiempo, el final no tiene sentido.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Recomiendo verla a todo el mundo, me hizo sentir feliz.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [23]:
df_sample_2 = iterate_classification(df, classify_slm_2, model, tokenizer, device, n_samples=1000)

Iniciando la predicción del SLM para 1000 muestras...


100%|██████████| 1000/1000 [18:43<00:00,  1.12s/it]


Y a la función para evaluar los resultados:

In [24]:
results_2 = evaluate_slm_predictions(df_sample_2)

Total de muestras evaluadas: 1000
Muestras con FALLO de instrucción ('INDEFINIDO'): 107 (10.70%)
-----------------------------------
Precisión (Accuracy): 0.8029
F1-score ponderado: 0.8012

Reporte completo:
              precision    recall  f1-score   support

    negative     0.9216    0.6897    0.7890       477
    positive     0.7239    0.9327    0.8151       416

    accuracy                         0.8029       893
   macro avg     0.8227    0.8112    0.8020       893
weighted avg     0.8295    0.8029    0.8012       893



Podemos ver que al agregar más ejemplos, el modelo clasifica mejor. Podemos ver también lo que sucede si no agregamos ejemplos:

In [25]:
def classify_slm_3(text_input, model, tokenizer, device):
    # Diseñamos el prompt, sin ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [26]:
df_sample_3 = iterate_classification(df, classify_slm_3, model, tokenizer, device, n_samples=1000)

Iniciando la predicción del SLM para 1000 muestras...


100%|██████████| 1000/1000 [14:40<00:00,  1.14it/s]


Y a la función para evaluar los resultados:

In [27]:
results_3 = evaluate_slm_predictions(df_sample_3)

Total de muestras evaluadas: 1000
Muestras con FALLO de instrucción ('INDEFINIDO'): 854 (85.40%)
-----------------------------------
Precisión (Accuracy): 0.4521
F1-score ponderado: 0.2815

Reporte completo:
              precision    recall  f1-score   support

    negative     0.0000    0.0000    0.0000        80
    positive     0.4521    1.0000    0.6226        66

    accuracy                         0.4521       146
   macro avg     0.2260    0.5000    0.3113       146
weighted avg     0.2044    0.4521    0.2815       146



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Podemos ver que al no agregar los ejemplos, las métricas decrecen muchísimo, lo que nos muestra la importancia de los ejemplos en el prompt.

Ahora cambiemos algunos parámetros. Por ejemplo, cambiamos `num_beams` de 5 a 10. Esto es, ahora se consideran las 10 secuencias más probables en cada paso, teniendo mayor robustez en la clasificación pero haciendo más lento el entrenamiento.

In [28]:
def classify_slm_4(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS EXTENDIDOS (4 Ejemplos) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"

            "Comentario: 'Una pérdida de tiempo, el final no tiene sentido.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Recomiendo verla a todo el mundo, me hizo sentir feliz.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [29]:
df_sample_4 = iterate_classification(df, classify_slm_4, model, tokenizer, device, n_samples=1000)

Iniciando la predicción del SLM para 1000 muestras...


100%|██████████| 1000/1000 [36:30<00:00,  2.19s/it]


Y a la función para evaluar los resultados:

In [30]:
results_4 = evaluate_slm_predictions(df_sample_4)

Total de muestras evaluadas: 1000
Muestras con FALLO de instrucción ('INDEFINIDO'): 107 (10.70%)
-----------------------------------
Precisión (Accuracy): 0.8029
F1-score ponderado: 0.8012

Reporte completo:
              precision    recall  f1-score   support

    negative     0.9216    0.6897    0.7890       477
    positive     0.7239    0.9327    0.8151       416

    accuracy                         0.8029       893
   macro avg     0.8227    0.8112    0.8020       893
weighted avg     0.8295    0.8029    0.8012       893



Podemos ver que los resultados son los mismos que para `num_beams=5`, por lo que nos quedamos con ese valor.

Por último, cambiemos el `max_new_tokens` de 2 a 5, buscando mejorar la robustez de la clasificación al darle al modelo más margen para generar la etiqueta completa, lo que es crucial para reducir los fallos 'INDEFINIDO'.

In [31]:
def classify_slm_5(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS EXTENDIDOS (4 Ejemplos) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"

            "Comentario: 'Una pérdida de tiempo, el final no tiene sentido.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Recomiendo verla a todo el mundo, me hizo sentir feliz.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [32]:
df_sample_5 = iterate_classification(df, classify_slm_5, model, tokenizer, device, n_samples=1000)

Iniciando la predicción del SLM para 1000 muestras...


100%|██████████| 1000/1000 [23:01<00:00,  1.38s/it]


Y a la función para evaluar los resultados:

In [33]:
results_5 = evaluate_slm_predictions(df_sample_5)

Total de muestras evaluadas: 1000
Muestras con FALLO de instrucción ('INDEFINIDO'): 88 (8.80%)
-----------------------------------
Precisión (Accuracy): 0.7840
F1-score ponderado: 0.7804

Reporte completo:
              precision    recall  f1-score   support

    negative     0.9286    0.6433    0.7600       485
    positive     0.6997    0.9438    0.8036       427

    accuracy                         0.7840       912
   macro avg     0.8141    0.7935    0.7818       912
weighted avg     0.8214    0.7840    0.7804       912



## Comparación de todos los modelos

In [34]:
# Creamos un diccionario
precision_comparison = {
    'Method': ['Initial Prompt (2 Examples)', 'Extended Prompt (4 Examples)', 'No Examples', 'Extended Prompt (num_beams=10)', 'Extended Prompt (max_new_tokens=5)'],
    'Accuracy': [results['accuracy'], results_2['accuracy'], results_3['accuracy'], results_4['accuracy'], results_5['accuracy']],
    'F1-score (weighted)': [results['f1_weighted'], results_2['f1_weighted'], results_3['f1_weighted'], results_4['f1_weighted'], results_5['f1_weighted']],
    'Failed Samples (%)': [results['failed_samples']/results['total_samples']*100,
                           results_2['failed_samples']/results_2['total_samples']*100,
                           results_3['failed_samples']/results_3['total_samples']*100,
                           results_4['failed_samples']/results_4['total_samples']*100,
                           results_5['failed_samples']/results_5['total_samples']*100]
}

# Lo pasamos a un df
df_comparison = pd.DataFrame(precision_comparison)

# Mostramos la tabla de comparación
print("--- Comparación de Resultados de Clasificación del SLM ---")
display(df_comparison)

--- Comparación de Resultados de Clasificación del SLM ---


,Method,Accuracy,F1-score (weighted),Failed Samples (%)
0,Initial Prompt (2 Examples),0.792265,0.789479,9.5
1,Extended Prompt (4 Examples),0.802912,0.801154,10.7
2,No Examples,0.452055,0.281468,85.4
3,Extended Prompt (num_beams=10),0.802912,0.801154,10.7
4,Extended Prompt (max_new_tokens=5),0.783991,0.780434,8.8


## Conclusión

Podemos comparar lo realizado en el trabajo anterior con el modelo clásico basado en TF-IDF y el SLM Qwen-0.6B (mediante Prompt Engineering).

El modelo de TF-IDF con un clasificador tradicional se confirma como el claro ganador en rendimiento puro, logrando una Precision de $\approx0.90$ y un F1-Score de $\approx0.89$.

El SLM Qwen-0.6B demostro un rendimiento muy bueno, alcanzando un Accuracy maximo de $0.8029$ (con el Extended Prompt), a pesar de haber recibido cero entrenamiento especifico (fine-tuning) en el dataset. Observamos lo importante que es pasar de un Zero-Shot a un Few-Shot Prompting, agregando ejemplos. Esto nos dice que si seguimos mejorando el prompt, capaz podemos llegar a mejores resultados.

Los ajustes como el aumento del numero de ejemplos (Extended Prompt) y el margen de error (max_new_tokens=5) mejoraron la robustez, mientras que la iteracion de num_beams confirmo que 5 es un valor eficiente para este SLM.

Por lo tanto, en este caso, el SLM base no es mejor que el modelo tradicional de TF-IDF.

Sin embargo, el 80% de Accuracy obtenido mediante solo Prompt Engineering es una prueba del poder de su comprension contextual general.

Para que el SLM supere al modelo de TF-IDF, el siguiente paso debe ser la especializacion a traves del fine-tuning. Al entrenar el Qwen-0.6B con el dataset de IMDB, se eliminarán los fallos de instruccion y se adapptaría el modelo a la base de datos particular, seguramente maximizando el rendimiento del modelo.